# Code converter - Python to PHP Code

This implementation coverts a python code to optimized Nodejs Code.    

# Imports packages needed

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display, display_markdown
from system_info import retrieve_system_info
import gradio as gr


# Load environment variables

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    print(f"OPENAI_API_KEY environment variable is not set.")
else:
    print(f"OPENAI_API_KEY environment variable is set and it begins with: {openai_api_key[:8]}...")
    

# Initialize openai client

In [ ]:
openai_client = OpenAI()

MODEL =  "gpt-5-nano"


In [ ]:
# System info
system_info = retrieve_system_info()
display_markdown(f"## System Information\n```\n{system_info}\n```", raw=True)

In [ ]:
message = f"""
This is a report of system information for my machine.
I want to run a NodeJs compiler to compile a single elixir file called converted.js and then execute it in the easiest way possible.
Please reply with if I need to install any elixir compiler to be make this possible,
provide simple step by step instructions to do that.

if I'm already set up to compile and execute elixir files, then I'd like something like this in python to compile and execute the code
```python
compile_command = # something here - to achieve a good runtime performance
compile_result = subprocess.run(compile_command, check=True, capture_output=True)
run_command = # something here - to achieve a good runtime performance
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout

 Do not forget to let me know what I should use for the compile_command ad run_command.

 System information:
{system_info}
"""

In [ ]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": message}]
)
display(Markdown(response.choices[0].message.content))


In [ ]:
compile_command = ["elixirc", "converted.ex"]
run_command = ["elixir", "converted.ex"]

## System and user prompts for coder conversion

In [ ]:
# system prompt
system_prompt = """
You are an expert in software development,
Your task is to convert Python code into high performance elixir code.
Response only with Exlixir code. Do not provide any explanation other than occasional comments in the code itself.
The Elixir code response should produce an identical output in the fastest possble time.
"""

def user_prompt(pyhon_code):
    return f"""
    Port this Python ode to Elixir with the highest possible performance. the elixir code should produce the same output as the python code.

    The system information is
    {system_info}

    Your response will be written to a file called converted.ex and then compile and execute; the compile command is:
    {compile_command}

    Respond only with Elixir code.
    Python code to port:

    ```python
    {pyhon_code}
    ```
    """


In [ ]:
def message(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(python_code)}
    ]

In [ ]:
def write_output(code):
    with open("converted.ex", "w", encoding="utf-8") as f:
        f.write(code)

In [ ]:
def convert(python_code):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=message(python_code),
        reasoning_effort="high",
    )
    reply = response.choices[0].message.content
    reply = reply.replace("```elixir", "").replace("```", "")
    return reply


In [ ]:
python_code = """
import time
 
def fibonacci(n):
    if n <= 0:
         return []
    elif n == 1:
        return [0]
    
    sequence = [0, 1]
    while len(sequence) < n:
        sequence.append(sequence[-1] + sequence[-2])
    return sequence

# Recursive version
def fibonacci_recursive(n):
    if n <= 1:
      return n
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)

# Sample usage
start_time = time.time()
result1 = fibonacci(30)
end_time = time.time()
print(f"Iterative version took {end_time - start_time:.6f} seconds")
print(result1)

start_time = time.time()
result2 = fibonacci_recursive(30)
end_time = time.time()
print(f"Recursive version took {end_time - start_time:.6f} seconds")
print(result2)
        
 """

In [ ]:
def run_python_code(python_code):
    globals_dict = {"__bultins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(python_code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error executing Python code: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# Run 
run_python_code(python_code)

In [ ]:
# Convert
convert(python_code)

In [ ]:
def run_converted_code(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"Error running converted code: {e.stderr.decode()}"


# Load in gradio UI


In [ ]:
with gr.Blocks(
    theme=gr.themes.Monochrome(),
    title="Port From PPython To Elixir",
) as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python Original Code",
                value=python_code,
                language="python"
                lines=50,
            )
        with gr.Column(scale=6):
            elixir = gr.Code(
                label="Elixir Converted Code",
                value="",
                language="elixir"
                lines=50,
            )
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python Code", elem_classes=["run-btn", "py"])
        port = gr.Button("Convert to Elixir", elem_classes=["convert-btn"])
        elixir_run = gr.Button("Run Elixir Code", elem_classes=["run-btn", "ex"])

    with gr.Row(equal_height=True):
        
        with gr.Column(scale=6):
            python_output = gr.Textbox(
                label="Python Output",
                value="",
                lines=10,
            )
        with gr.Column(scale=6):
            elixir_output = gr.Textbox(
                label="Elixir Output",
                value="",
                lines=10,
            )
    port.click(
        fn=convert,
        inputs=[python],
        outputs=[elixir],
    )
    python_run.click(
        fn=run_python_code,
        inputs=[python],
        outputs=[python_output],
    )
    elixir_run.click(
        fn=run_converted_code,
        inputs=[elixir],
        outputs=[elixir_output],
    )

ui.launch(inbrowser=True)            